In [10]:
import os
import sys

%load_ext autoreload
%autoreload 2
import logging

import pandas as pd
import numpy as np
import scipy.stats
import matplotlib.pyplot as plt


# Configure logging
logging.basicConfig(level=logging.INFO)

# If your current working directory is the notebooks directory, use this:
notebook_dir = os.getcwd()  # current working directory
src_path = os.path.abspath(os.path.join(notebook_dir, "..", "src"))
sys.path.append(src_path)

# Add the parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(notebook_dir, ".."))
sys.path.append(parent_dir)
import pickle
from server_config import (
    datapath,
    preprocessed_path_freezed,
    redcap_path,
    preprocessed_path,
)
from functions.preprocessing.aggregation import compute_sleep_sessions


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
# backup_path = preprocessed_path_freezed + "/backup_data_passive_actual.feather"
backup_path = (
    "/sc-projects/sc-proj-cc15-preact/SP6/raw/backup_passive_recent.feather"  # new file
)
df_backup = pd.read_feather(backup_path)
print(df_backup.shape)
df_backup.head()

(119504032, 16)


,customer,for_id,type,startTimestamp,endTimestamp,start_end,doubleValue,longValue,booleanValue,startTimestamp_day,startTimestamp_hour,ema_base_start,study_version,createdAt,inferred_tzoffset,inferred_tzoffset_source
0,4MLe,FOR11905,Steps,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,60.0,6.00,NaN,<NA>,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine
1,4MLe,FOR11905,ActiveBurnedCalories,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,60.0,0.14,NaN,<NA>,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine
2,4MLe,FOR11905,CoveredDistance,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,60.0,4.62,NaN,<NA>,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine
3,4MLe,FOR11905,HeartRate,2023-05-17 14:58:01+00:00,2023-05-17 14:58:38+00:00,37.0,NaN,74.0,<NA>,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine
4,4MLe,FOR11905,HeartRate,2023-05-17 18:00:55+00:00,NaT,NaN,NaN,95.0,<NA>,2023-05-17 00:00:00+00:00,18.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine


In [12]:
# TODO move it to 01_data_preprocess, as it takes some time to run, and makes more sense there

df_backup["local_start_time"] = (
    df_backup["startTimestamp"]
    + pd.to_timedelta(df_backup["inferred_tzoffset"], unit="min")
).dt.tz_localize(None)

df_backup["local_end_time"] = (
    df_backup["endTimestamp"]
    + pd.to_timedelta(df_backup["inferred_tzoffset"], unit="min")
).dt.tz_localize(None)

df_backup.head()

,customer,for_id,type,startTimestamp,endTimestamp,start_end,doubleValue,longValue,booleanValue,startTimestamp_day,startTimestamp_hour,ema_base_start,study_version,createdAt,inferred_tzoffset,inferred_tzoffset_source,local_start_time,local_end_time
0,4MLe,FOR11905,Steps,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,60.0,6.00,NaN,<NA>,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine,2023-05-17 15:44:00,2023-05-17 15:45:00
1,4MLe,FOR11905,ActiveBurnedCalories,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,60.0,0.14,NaN,<NA>,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine,2023-05-17 15:44:00,2023-05-17 15:45:00
2,4MLe,FOR11905,CoveredDistance,2023-05-17 14:44:00+00:00,2023-05-17 14:45:00+00:00,60.0,4.62,NaN,<NA>,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine,2023-05-17 15:44:00,2023-05-17 15:45:00
3,4MLe,FOR11905,HeartRate,2023-05-17 14:58:01+00:00,2023-05-17 14:58:38+00:00,37.0,NaN,74.0,<NA>,2023-05-17 00:00:00+00:00,14.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine,2023-05-17 15:58:01,2023-05-17 15:58:38
4,4MLe,FOR11905,HeartRate,2023-05-17 18:00:55+00:00,NaT,NaN,NaN,95.0,<NA>,2023-05-17 00:00:00+00:00,18.0,2023-05-17 00:00:00+00:00,Lang,NaT,60.0,interpolate_no_previous_next_is_fine,2023-05-17 19:00:55,NaT


# Sleep

In [13]:
df_sleep_sessions = compute_sleep_sessions(df_backup)
df_sleep_sessions.head()

/home/milu10/src/tiki_code/functions/preprocessing/aggregation.py:54: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sleepall.groupby("customer")["is_lastentryinsession"].shift(1).fillna(True)


,customer,sleep_session_id,startTimestamp,endTimestamp,local_start_time,local_end_time,sleep_session_duration,SleepAwake_duration,SleepDeep_duration,SleepInBed_duration,...,time_in_bed,time_out_of_bed,sleep_efficiency,hypersomnia,insomnia,awake_pct,light_sleep_pct,deep_sleep_pct,day,num_sessions_in_day
0,05kz,0,2023-10-10 22:13:00+00:00,2023-10-11 06:00:00+00:00,2023-10-11 00:13:00,2023-10-11 08:00:00,28020.0,780.0,15060.0,27240.0,...,28020.0,0.0,0.972163,False,False,0.027837,0.434690,0.537473,2023-10-11,1
1,05kz,1,2023-10-11 21:11:00+00:00,2023-10-12 08:00:00+00:00,2023-10-11 23:11:00,2023-10-12 10:00:00,38940.0,5160.0,18600.0,33780.0,...,38940.0,0.0,0.867488,False,False,0.132512,0.391371,0.477658,2023-10-12,1
2,05kz,2,2023-10-12 21:10:00+00:00,2023-10-13 06:14:00+00:00,2023-10-12 23:10:00,2023-10-13 08:14:00,32640.0,1980.0,20280.0,30660.0,...,32640.0,0.0,0.939338,False,False,0.060662,0.318015,0.621324,2023-10-13,1
3,05kz,3,2023-10-14 23:48:00+00:00,2023-10-15 04:30:00+00:00,2023-10-15 01:48:00,2023-10-15 06:30:00,16920.0,120.0,8580.0,16800.0,...,16920.0,0.0,0.992908,False,False,0.007092,0.485816,0.507092,2023-10-15,1
4,05kz,4,2023-10-15 21:11:00+00:00,2023-10-16 07:28:00+00:00,2023-10-15 23:11:00,2023-10-16 09:28:00,37020.0,1380.0,17881.0,35640.0,...,37020.0,0.0,0.962723,False,False,0.037277,0.479714,0.483009,2023-10-16,1


more than 90% of the customer-days got only one session, but some have more, what to do later?
- only analyze one, longest sleep session
- aggregate sessions further by summing over relevant columns

In [14]:
# new df with just longest sleep session per customer per day
df_sleep_sessions_longest = df_sleep_sessions.loc[
    df_sleep_sessions.groupby(["customer", "day"])["sleep_session_duration"].idxmax()
].reset_index(drop=True)
# df_sleep_sessions_longest.groupby(["customer", "day"]).size().value_counts()
df_sleep_sessions_longest.head()

,customer,sleep_session_id,startTimestamp,endTimestamp,local_start_time,local_end_time,sleep_session_duration,SleepAwake_duration,SleepDeep_duration,SleepInBed_duration,...,time_in_bed,time_out_of_bed,sleep_efficiency,hypersomnia,insomnia,awake_pct,light_sleep_pct,deep_sleep_pct,day,num_sessions_in_day
0,05kz,0,2023-10-10 22:13:00+00:00,2023-10-11 06:00:00+00:00,2023-10-11 00:13:00,2023-10-11 08:00:00,28020.0,780.0,15060.0,27240.0,...,28020.0,0.0,0.972163,False,False,0.027837,0.434690,0.537473,2023-10-11,1
1,05kz,1,2023-10-11 21:11:00+00:00,2023-10-12 08:00:00+00:00,2023-10-11 23:11:00,2023-10-12 10:00:00,38940.0,5160.0,18600.0,33780.0,...,38940.0,0.0,0.867488,False,False,0.132512,0.391371,0.477658,2023-10-12,1
2,05kz,2,2023-10-12 21:10:00+00:00,2023-10-13 06:14:00+00:00,2023-10-12 23:10:00,2023-10-13 08:14:00,32640.0,1980.0,20280.0,30660.0,...,32640.0,0.0,0.939338,False,False,0.060662,0.318015,0.621324,2023-10-13,1
3,05kz,3,2023-10-14 23:48:00+00:00,2023-10-15 04:30:00+00:00,2023-10-15 01:48:00,2023-10-15 06:30:00,16920.0,120.0,8580.0,16800.0,...,16920.0,0.0,0.992908,False,False,0.007092,0.485816,0.507092,2023-10-15,1
4,05kz,4,2023-10-15 21:11:00+00:00,2023-10-16 07:28:00+00:00,2023-10-15 23:11:00,2023-10-16 09:28:00,37020.0,1380.0,17881.0,35640.0,...,37020.0,0.0,0.962723,False,False,0.037277,0.479714,0.483009,2023-10-16,1


In [15]:
df_sleep_sessions.groupby(["customer", "day"]).agg(
    {
        "sleep_session_duration": "sum",
        "total_sleep_time": "sum",
        "time_in_bed": "sum",
        "time_out_of_bed": "sum",
        "SleepAwake_duration": "sum",
        "SleepLight_duration": "sum",
        "SleepDeep_duration": "sum",
        "awakenings": "sum",
        "long_awakenings": "sum",
        "num_sessions_in_day": "first",  # same for all sessions in that day
    }
)

sleep_session_duration  total_sleep_time  time_in_bed  \
customer day                                                                 
05kz     2023-10-11                 28020.0           27240.0      28020.0   
         2023-10-12                 38940.0           33780.0      38940.0   
         2023-10-13                 32640.0           30660.0      32640.0   
         2023-10-15                 16920.0           16800.0      16920.0   
         2023-10-16                 37020.0           35640.0      37020.0   
...                                     ...               ...          ...   
zgxc     2024-06-22                 31620.0           30120.0      31620.0   
         2024-06-23                 30780.0           29820.0      30780.0   
         2024-06-24                 28980.0           27660.0      28980.0   
         2024-06-25                 21000.0           20460.0      21000.0   
         2024-06-26                 30300.0           29460.0      30300.0   

                     time_out_of_bed  SleepAwake_duration  \
customer day                                                
05kz     2023-10-11              0.0                780.0   
         2023-10-12              0.0               5160.0   
         2023-10-13              0.0               1980.0   
         2023-10-15              0.0                120.0   
         2023-10-16              0.0               1380.0   
...                              ...                  ...   
zgxc     2024-06-22              0.0               1500.0   
         2024-06-23              0.0                960.0   
         2024-06-24              0.0               1320.0   
         2024-06-25              0.0                540.0   
         2024-06-26              0.0                840.0   

                     SleepLight_duration  SleepDeep_duration  awakenings  \
customer day                                                               
05kz     2023-10-11              12180.0             15060.0         1.0   
         2023-10-12              15240.0             18600.0         3.0   
         2023-10-13              10380.0             20280.0         2.0   
         2023-10-15               8220.0              8580.0         0.0   
         2023-10-16              17759.0             17881.0         3.0   
...                                  ...                 ...         ...   
zgxc     2024-06-22              21480.0              8640.0         2.0   
         2024-06-23              24000.0              5820.0         1.0   
         2024-06-24              20640.0              7020.0         2.0   
         2024-06-25              15720.0              4740.0         1.0   
         2024-06-26              20040.0              9420.0         1.0   

                     long_awakenings  num_sessions_in_day  
customer day                                               
05kz     2023-10-11              0.0                    1  
         2023-10-12              1.0                    1  
         2023-10-13              0.0                    1  
         2023-10-15              0.0                    1  
         2023-10-16              0.0                    1  
...                              ...                  ...  
zgxc     2024-06-22              0.0                    1  
         2024-06-23              0.0                    1  
         2024-06-24              0.0                    1  
         2024-06-25              0.0                    1  
         2024-06-26              0.0                    1  

[88324 rows x 10 columns]